In [49]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    median_absolute_error,
    r2_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from collections import defaultdict

BASE_DIR = Path(".")

SENTINEL = -1.0

VENTANA_MIN = 60
PASO_MIN = 5
HORIZONTE_MIN = 120

In [10]:
def infer_optimal_dtypes(
    csv_path,
    chunksize=100_000,
    max_rows=None,
    low_memory=False
):
    """
    First pass over CSV:
    - Reads in chunks
    - Detects optimal dtype for each column
    - Handles ints, floats, bools, strings, categories, datetimes

    Returns:
        dtype_map: dict for pandas dtype loading
        parse_dates: list of datetime columns
    """

    stats = defaultdict(lambda: {
        "dtype_candidates": set(),
        "min": None,
        "max": None,
        "n_unique_sample": set(),
        "non_null": 0,
    })

    rows_read = 0

    reader = pd.read_csv(
        csv_path,
        chunksize=chunksize,
        low_memory=low_memory
    )

    for chunk in reader:

        if max_rows is not None:
            remaining = max_rows - rows_read
            if remaining <= 0:
                break
            chunk = chunk.iloc[:remaining]

        rows_read += len(chunk)

        for col in chunk.columns:
            s = chunk[col].dropna()

            if len(s) == 0:
                continue

            info = stats[col]

            # Try bool detection
            unique_vals = set(s.unique())

            bool_like = unique_vals.issubset(
                {0, 1, True, False, "0", "1", "True", "False", "true", "false"}
            )

            if bool_like:
                info["dtype_candidates"].add("bool")
                continue

            # Integer columns
            if pd.api.types.is_integer_dtype(s):

                info["dtype_candidates"].add("int")

                col_min = s.min()
                col_max = s.max()

                info["min"] = (
                    col_min if info["min"] is None
                    else min(info["min"], col_min)
                )

                info["max"] = (
                    col_max if info["max"] is None
                    else max(info["max"], col_max)
                )

                continue

            # Float columns
            if pd.api.types.is_float_dtype(s):

                info["dtype_candidates"].add("float")

                col_min = s.min()
                col_max = s.max()

                info["min"] = (
                    col_min if info["min"] is None
                    else min(info["min"], col_min)
                )

                info["max"] = (
                    col_max if info["max"] is None
                    else max(info["max"], col_max)
                )

                continue

            # Datetime detection (sample-based)
            if len(s) > 0:
                sample = s.iloc[:50]

                try:
                    parsed = pd.to_datetime(sample, errors="raise")

                    if parsed.notna().mean() > 0.95:
                        info["dtype_candidates"].add("datetime")
                        continue

                except Exception:
                    pass

            # String / category detection
            info["dtype_candidates"].add("object")

            # Sample uniques for category decision
            sample_uniques = set(s.astype(str).unique()[:5000])
            info["n_unique_sample"].update(sample_uniques)
            info["non_null"] += len(s)

    # Decide final dtypes
    dtype_map = {}
    parse_dates = []

    for col, info in stats.items():

        candidates = info["dtype_candidates"]

        # Boolean
        if candidates == {"bool"}:
            dtype_map[col] = "boolean"
            continue

        # Integer
        if "int" in candidates and "float" not in candidates:

            min_val = info["min"]
            max_val = info["max"]

            if min_val >= 0:
                if max_val <= np.iinfo(np.uint8).max:
                    dtype_map[col] = np.uint8
                elif max_val <= np.iinfo(np.uint16).max:
                    dtype_map[col] = np.uint16
                elif max_val <= np.iinfo(np.uint32).max:
                    dtype_map[col] = np.uint32
                else:
                    dtype_map[col] = np.uint64

            else:
                if (
                    min_val >= np.iinfo(np.int8).min
                    and max_val <= np.iinfo(np.int8).max
                ):
                    dtype_map[col] = np.int8

                elif (
                    min_val >= np.iinfo(np.int16).min
                    and max_val <= np.iinfo(np.int16).max
                ):
                    dtype_map[col] = np.int16

                elif (
                    min_val >= np.iinfo(np.int32).min
                    and max_val <= np.iinfo(np.int32).max
                ):
                    dtype_map[col] = np.int32

                else:
                    dtype_map[col] = np.int64

            continue

        # Float
        if "float" in candidates:
            min_val = info["min"]
            max_val = info["max"]

            # Float32 unless too large
            float32_limit = np.finfo(np.float32).max

            if (
                abs(min_val) < float32_limit
                and abs(max_val) < float32_limit
            ):
                dtype_map[col] = np.float32
            else:
                dtype_map[col] = np.float64

            continue

        # Datetime
        if "datetime" in candidates:
            parse_dates.append(col)
            continue

        # String/category
        if "object" in candidates:

            unique_count = len(info["n_unique_sample"])
            total_count = max(info["non_null"], 1)

            # Heuristic:
            # low cardinality => category
            if unique_count / total_count < 0.2:
                dtype_map[col] = "category"
            else:
                dtype_map[col] = "string"

    return dtype_map, parse_dates


def load_csv_optimized(
    csv_path,
    chunksize=100_000,
    max_rows=None
):
    """
    Optimized CSV loading using inferred dtypes.

    Returns:
        pandas DataFrame
    """

    print("Inferring dtypes...")
    dtype_map, parse_dates = infer_optimal_dtypes(
        csv_path,
        chunksize=chunksize,
        max_rows=max_rows
    )

    print("Chosen dtypes:")
    for k, v in dtype_map.items():
        print(f"{k}: {v}")

    chunks = []
    rows_read = 0

    reader = pd.read_csv(
        csv_path,
        chunksize=chunksize,
        dtype=dtype_map,
        parse_dates=parse_dates,
        low_memory=False
    )

    for chunk in reader:

        if max_rows is not None:
            remaining = max_rows - rows_read
            if remaining <= 0:
                break
            chunk = chunk.iloc[:remaining]

        rows_read += len(chunk)
        chunks.append(chunk)

    df = pd.concat(chunks, ignore_index=True)

    return df

In [22]:
def save_dataframe_parquet(
    df: pd.DataFrame,
    output_path="dataset.parquet",
    compression="snappy",
    index=False
):
    """
    Save a DataFrame to Parquet.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to save.

    output_path : str
        Output parquet file path.

    compression : str
        Compression type:
        - 'snappy' (fast, default)
        - 'gzip'   (smaller file, slower)
        - 'brotli' (very compact)
        - 'zstd'   (excellent balance)
        - None     (no compression)

    index : bool
        Whether to save DataFrame index.
    """

    df.to_parquet(
        output_path,
        engine="pyarrow",   # recommended
        compression=compression,
        index=index
    )

    print(f"Saved parquet to: {output_path}")

def load_parquet_df(parquet_path, columns=None):
    """
    Load a DataFrame from parquet.

    Parameters
    ----------
    parquet_path : str
        Path to parquet file.

    columns : list[str] | None
        Optional subset of columns to load.

    Returns
    -------
    pd.DataFrame
    """

    df = pd.read_parquet(
        parquet_path,
        engine="pyarrow",
        columns=columns
    )

    print(f"Loaded dataframe: {df.shape}")
    return df

In [11]:
def limpiar_sentinels(df, sentinel=SENTINEL):
    cols_numericas = df.select_dtypes(include="number").columns
    df[cols_numericas] = df[cols_numericas].replace(sentinel, np.nan)
    return df


def cargar_dataset(nombre_archivo):
    df = load_csv_optimized(nombre_archivo)
    df = limpiar_sentinels(df)

    return (
        df
        .sort_values(["source", "TIME"])
        .reset_index(drop=True)
    )


def detectar_sensores_disponibles(*dfs):
    cols = set().union(*(set(df.columns) for df in dfs))

    return sorted(
        {
            int(m.group(1))
            for c in cols
            if (m := re.match(r"ox_s(\d+)", c))
        }
    )

In [12]:
def calcular_tiempo_hasta_setpoint(
    df,
    sensor=1,
    horizonte_min=360,
):

    ox_col = f"ox_s{sensor}"
    sp_col = f"sp_s{sensor}"

    trabajo = df[
        ["source", "TIME", "suma_psa", ox_col, sp_col]
    ].copy()

    trabajo["psa_on"] = (
        trabajo["suma_psa"].fillna(0) > 0
    )

    eventos = []

    for source, chunk in trabajo.groupby(
        "source",
        sort=False
    ):

        chunk = chunk.reset_index(drop=True)

        inicio_mask = (
            chunk["psa_on"]
            &
            ~chunk["psa_on"].shift(fill_value=False)
        )

        for idx in chunk.index[inicio_mask]:

            inicio = chunk.loc[idx, "TIME"]
            ox_inicio = chunk.loc[idx, ox_col]

            ventana = chunk[
                (chunk["TIME"] >= inicio)
                &
                (
                    chunk["TIME"]
                    <= inicio
                    + pd.Timedelta(minutes=horizonte_min)
                )
            ].copy()

            setpoint = (
                ventana[sp_col]
                .dropna()
                .median()
            )

            if pd.isna(setpoint):
                continue

            if pd.isna(ox_inicio):
                continue

            if ox_inicio >= setpoint:
                continue

            ventana["target"] = (
                ventana[sp_col]
                .fillna(setpoint)
            )

            cruce = ventana[
                ventana[ox_col]
                >= ventana["target"]
            ]

            minutos = (
                np.nan
                if cruce.empty
                else (
                    cruce.iloc[0]["TIME"]
                    - inicio
                ).total_seconds()
                / 60
            )

            eventos.append(
                {
                    "source": source,
                    "inicio": inicio,
                    "sensor": sensor,
                    "setpoint": setpoint,
                    "ox_inicio": ox_inicio,
                    "minutos_hasta_setpoint": minutos,
                }
            )

    return pd.DataFrame(eventos)

In [13]:
FEATURE_COLS = [
    "ox_mean",
    "ox_std",
    "ox_min",
    "ox_max",
    "ox_last",
    "ox_slope",
    "sp_mean",
    "gap_ox_sp",
    "psa_ratio",
    "n_arranques_previos",
    "cobertura_ox",
    "hora_del_dia",
    "dia_semana",
]

In [14]:
def construir_features_ventana(
    df,
    eventos,
    sensor,
    ventana_min=60,
    paso_min=5,
):

    ox_col = f"ox_s{sensor}"
    sp_col = f"sp_s{sensor}"

    registros = []

    for _, ev in eventos.iterrows():

        source = ev["source"]
        inicio = ev["inicio"]

        chunk = (
            df[df["source"] == source]
            .sort_values("TIME")
        )

        ventana = chunk[
            (
                chunk["TIME"]
                >= inicio
                - pd.Timedelta(minutes=ventana_min)
            )
            &
            (
                chunk["TIME"]
                < inicio
            )
        ].copy()

        if ventana.empty:
            continue

        ventana = ventana.set_index("TIME")

        ventana_rs = (
            ventana[
                [ox_col, sp_col, "suma_psa"]
            ]
            .resample(f"{paso_min}min")
            .mean()
        )

        ox = ventana_rs[ox_col].dropna()

        if len(ox) < 3:
            continue

        sp_vals = ventana_rs[sp_col].dropna()
        psa_vals = (
            ventana_rs["suma_psa"]
            .fillna(0)
            .gt(0)
            .astype(int)
        )

        x = np.arange(len(ox))

        slope = (
            np.polyfit(
                x,
                ox.values,
                1
            )[0]
        )

        psa_diff = psa_vals.diff().fillna(0)

        feat = {
            "grupo": ev["grupo"],
            "source": source,
            "inicio": inicio,
            "sensor": sensor,

            "ox_mean": ox.mean(),
            "ox_std": ox.std(ddof=0),
            "ox_min": ox.min(),
            "ox_max": ox.max(),
            "ox_last": ox.iloc[-1],
            "ox_slope": slope,

            "sp_mean": sp_vals.mean(),
            "gap_ox_sp": (
                ev["setpoint"]
                - ev["ox_inicio"]
            ),

            "psa_ratio": psa_vals.mean(),
            "n_arranques_previos": (
                psa_diff > 0
            ).sum(),

            "cobertura_ox": (
                len(ox)
                / len(ventana_rs)
            ),

            "hora_del_dia":
                inicio.hour
                + inicio.minute / 60,

            "dia_semana":
                inicio.dayofweek,

            "minutos_hasta_setpoint":
                ev["minutos_hasta_setpoint"],
        }

        registros.append(feat)

    return pd.DataFrame(registros)

In [ ]:
def entrenar_rf(
    df_feat,
    n_estimators=200,
    random_state=42,
):

    df_clean = (
        df_feat
        .dropna(subset=FEATURE_COLS)
        .copy()
    )

    X = df_clean[FEATURE_COLS]
    y = np.log1p(
        df_clean["minutos_hasta_setpoint"]
    )

    groups = df_clean["source"]

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)

if df_clean["source"].nunique() == 1:

    train_idx, test_idx = train_test_split(
        np.arange(len(X_scaled)),
        test_size=0.2,
        random_state=random_state,
    )

else:

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=random_state,
    )

    train_idx, test_idx = next(
        splitter.split(
            X_scaled,
            y,
            groups=df_clean["source"]
        )
    )

    X_train = X_scaled[train_idx]
    X_test = X_scaled[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=12,
        min_samples_leaf=4,
        n_jobs=-1,
        random_state=random_state,
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)

    metricas = {
        "mae":
            mean_absolute_error(
                y_test,
                y_pred,
            ),

        "medae":
            median_absolute_error(
                y_test,
                y_pred,
            ),

        "r2":
            r2_score(
                y_test,
                y_pred,
            ),
    }

    return rf, scaler, metricas

In [16]:
df_3comp = cargar_dataset(
    "dataset_3comp_limpio.csv"
)

df_4comp = cargar_dataset(
    "dataset_4comp_limpio.csv"
)

sensor_nums = detectar_sensores_disponibles(
    df_3comp,
    df_4comp,
)

df_todo = pd.concat(
    [df_3comp, df_4comp],
    ignore_index=True,
)

Inferring dtypes...


/tmp/ipykernel_79751/1611202921.py:108: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(sample, errors="raise")
/tmp/ipykernel_79751/1611202921.py:108: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(sample, errors="raise")
/tmp/ipykernel_79751/1611202921.py:108: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(sample, errors="raise")
/tmp/ipykernel_79751/1611202921.py:108: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consist

Chosen dtypes:
source: category
psi_psa1: <class 'numpy.float32'>
psi_psa2: <class 'numpy.float32'>
psi_psa3: <class 'numpy.float32'>
psi_psa4: <class 'numpy.float32'>
psi_tablero: <class 'numpy.float32'>
flujo: <class 'numpy.float32'>
totalizador: <class 'numpy.float32'>
r_psa1: <class 'numpy.int8'>
r_psa2: <class 'numpy.int8'>
r_psa3: <class 'numpy.int8'>
r_psa4: <class 'numpy.int8'>
r_gen1: <class 'numpy.int8'>
r_gen2: <class 'numpy.int8'>
r_bar: <class 'numpy.int8'>
r_sec1: <class 'numpy.int8'>
r_sec2: <class 'numpy.int8'>
r_com1: <class 'numpy.int8'>
r_com2: <class 'numpy.int8'>
r_com3: <class 'numpy.int8'>
sp_s1: <class 'numpy.float32'>
sp_s2: <class 'numpy.float32'>
sp_s3: <class 'numpy.float32'>
sp_s4: <class 'numpy.float32'>
sp_s5: <class 'numpy.float32'>
sp_s6: <class 'numpy.float32'>
sp_s7: <class 'numpy.float32'>
sp_s8: <class 'numpy.float32'>
sp_s9: <class 'numpy.float32'>
sp_s10: <class 'numpy.float32'>
sp_s11: <class 'numpy.float32'>
sp_s12: <class 'numpy.float32'>
hb_ge

/tmp/ipykernel_79751/1611202921.py:108: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(sample, errors="raise")
/tmp/ipykernel_79751/1611202921.py:108: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(sample, errors="raise")
/tmp/ipykernel_79751/1611202921.py:108: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(sample, errors="raise")
/tmp/ipykernel_79751/1611202921.py:108: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consist

Chosen dtypes:
source: category
psi_psa1: <class 'numpy.float32'>
psi_psa2: <class 'numpy.float32'>
psi_psa3: <class 'numpy.float32'>
psi_psa4: <class 'numpy.float32'>
psi_tablero: <class 'numpy.float32'>
flujo: <class 'numpy.float32'>
totalizador: <class 'numpy.float32'>
r_psa1: <class 'numpy.int8'>
r_psa2: <class 'numpy.int8'>
r_psa3: <class 'numpy.int8'>
r_psa4: <class 'numpy.int8'>
r_gen1: <class 'numpy.int8'>
r_gen2: <class 'numpy.int8'>
r_bar: <class 'numpy.int8'>
r_sec1: <class 'numpy.int8'>
r_sec2: <class 'numpy.int8'>
r_com1: <class 'numpy.int8'>
r_com2: <class 'numpy.int8'>
r_com3: <class 'numpy.int8'>
r_com4: <class 'numpy.int8'>
sp_s1: <class 'numpy.float32'>
sp_s2: <class 'numpy.float32'>
sp_s3: <class 'numpy.float32'>
sp_s4: <class 'numpy.float32'>
sp_s5: <class 'numpy.float32'>
sp_s6: <class 'numpy.float32'>
sp_s7: <class 'numpy.float32'>
sp_s8: <class 'numpy.float32'>
sp_s9: <class 'numpy.float32'>
sp_s10: <class 'numpy.float32'>
sp_s11: <class 'numpy.float32'>
sp_s12: 

In [17]:
eventos_lista = []

for sensor in sensor_nums:

    ev3 = calcular_tiempo_hasta_setpoint(
        df_3comp,
        sensor=sensor,
    )

    ev4 = calcular_tiempo_hasta_setpoint(
        df_4comp,
        sensor=sensor,
    )

    if not ev3.empty:
        eventos_lista.append(
            ev3.assign(grupo="3comp")
        )

    if not ev4.empty:
        eventos_lista.append(
            ev4.assign(grupo="4comp")
        )

eventos_all = pd.concat(
    eventos_lista,
    ignore_index=True,
)

In [24]:
save_features = False
load_features = True

In [ ]:
if save_features:

    feats_lista = []

    for sensor in sensor_nums:

        ev_sensor = eventos_all[
            eventos_all["sensor"] == sensor
        ]

        feats_lista.append(
            construir_features_ventana(
                df_todo,
                ev_sensor,
                sensor,
                ventana_min=60,
                paso_min=5,
            )
        )

    df_feat = pd.concat(
        feats_lista,
        ignore_index=True,
    )

    save_dataframe_parquet(df_feat,
        "features_random_forest_parallel.parquet"
    )

elif load_features:
    load_parquet_df("features_random_forest_parallel.parquet")

/home/valkyrie/Documents/UC/2026/2026-1/IIC2433/Proyecto/Repo/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1037: RuntimeWarning: overflow encountered in cast
  result = result.astype(dtype, copy=False)
/home/valkyrie/Documents/UC/2026/2026-1/IIC2433/Proyecto/Repo/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1037: RuntimeWarning: overflow encountered in cast
  result = result.astype(dtype, copy=False)


In [20]:
for grupo in ["3comp", "4comp"]:
    df_grupo = df_feat[df_feat["grupo"] == grupo]

    print(grupo)
    print("rows:", len(df_grupo))
    print(
        "NaN targets:",
        df_grupo["minutos_hasta_setpoint"].isna().sum()
    )

3comp
rows: 11695
NaN targets: 1964
4comp
rows: 26796
NaN targets: 6793


In [29]:
df_feat = df_feat.dropna(
    subset=FEATURE_COLS + ["minutos_hasta_setpoint"]
)

In [37]:
for col in df_3comp:
    print("\n", col)
    print(df_3comp[col].describe())


 source
count     1987226
unique         14
top         POX60
freq       163749
Name: source, dtype: object

 psi_psa1
count    1.985797e+06
mean     4.718779e+01
std      2.166773e+01
min     -1.798467e+01
25%      4.432352e+01
50%      5.626738e+01
75%      6.166278e+01
max      8.825545e+01
Name: psi_psa1, dtype: float64

 psi_psa2
count    1.985797e+06
mean     4.521910e+01
std      2.190686e+01
min     -1.798467e+01
25%      3.912392e+01
50%      5.428036e+01
75%      6.048072e+01
max      7.982150e+01
Name: psi_psa2, dtype: float64

 psi_psa3
count    1.985797e+06
mean     4.761237e+01
std      2.155785e+01
min     -1.798467e+01
25%      4.657886e+01
50%      5.586853e+01
75%      6.125668e+01
max      8.181577e+01
Name: psi_psa3, dtype: float64

 psi_psa4
count    1.985797e+06
mean     4.693872e+01
std      2.151630e+01
min     -1.798467e+01
25%      4.398269e+01
50%      5.533188e+01
75%      6.093034e+01
max      7.850891e+01
Name: psi_psa4, dtype: float64

 psi_tablero
count

In [36]:
for col in FEATURE_COLS:
    print("\n", col)
    print(df_clean[col].describe())


 ox_mean
count    2.971400e+04
mean    -1.030725e+23
std               inf
min     -1.531348e+27
25%      5.429361e+00
50%      6.003185e+00
75%      6.532407e+00
max      1.720356e+01
Name: ox_mean, dtype: float64

 ox_std
count    2.971400e+04
mean              inf
std               NaN
min      0.000000e+00
25%      1.300469e-01
50%      2.048774e-01
75%      3.278340e-01
max               inf
Name: ox_std, dtype: float64

 ox_min
count    2.971400e+04
mean    -1.339943e+24
std               inf
min     -1.990753e+28
25%      4.986057e+00
50%      5.601302e+00
75%      6.176477e+00
max      1.195579e+01
Name: ox_min, dtype: float64

 ox_max
count    2.971400e+04
mean     4.654470e+17
std               inf
min     -2.250151e+00
25%      5.833560e+00
50%      6.437066e+00
75%      6.976625e+00
max      6.915147e+21
Name: ox_max, dtype: float64

 ox_last
count    29714.000000
mean         5.703431
std          1.024556
min         -2.264724
25%          5.229170
50%          5.805191


/home/valkyrie/Documents/UC/2026/2026-1/IIC2433/Proyecto/Repo/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1037: RuntimeWarning: overflow encountered in cast
  result = result.astype(dtype, copy=False)
/home/valkyrie/Documents/UC/2026/2026-1/IIC2433/Proyecto/Repo/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/home/valkyrie/Documents/UC/2026/2026-1/IIC2433/Proyecto/Repo/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1037: RuntimeWarning: overflow encountered in cast
  result = result.astype(dtype, copy=False)
/home/valkyrie/Documents/UC/2026/2026-1/IIC2433/Proyecto/Repo/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1037: RuntimeWarning: overflow encountered in cast
  result = result.astype(dtype, copy=False)


In [39]:
np.isinf(df_feat[FEATURE_COLS]).sum()

ox_mean                0
ox_std                 2
ox_min                 0
ox_max                 0
ox_last                0
ox_slope               0
sp_mean                0
gap_ox_sp              0
psa_ratio              0
n_arranques_previos    0
cobertura_ox           0
hora_del_dia           0
dia_semana             0
dtype: int64

In [40]:
bad = df_feat[np.isinf(df_feat["ox_std"])]

bad[
    [
        "source",
        "inicio",
        "sensor",
        "ox_mean",
        "ox_std",
        "ox_min",
        "ox_max",
        "ox_last",
        "cobertura_ox",
    ]
]

,source,inicio,sensor,ox_mean,ox_std,ox_min,ox_max,ox_last,cobertura_ox
33972,POX33,2026-04-07 12:21:23.051,9,-1.531348e+27,inf,-1.990753e+28,6.915147e+21,5.063498,1.0
36758,POX33,2026-04-07 12:21:23.051,11,-1.531348e+27,inf,-1.990753e+28,6.915147e+21,5.428781,1.0


In [41]:
bad_mask = np.isinf(df_feat["ox_std"])

print("Rows to remove:", bad_mask.sum())

df_feat = df_feat.loc[~bad_mask].copy()

Rows to remove: 2


In [42]:
modelos = {}
predicciones = []

for grupo in ["3comp", "4comp"]:

    df_grupo = (
        df_feat[
            df_feat["grupo"] == grupo
        ]
        .copy()
    )

    print(
        f"\nTraining model for {grupo}"
    )

    rf, scaler, metricas = entrenar_rf(
        df_grupo
    )

    modelos[grupo] = {
        "rf": rf,
        "scaler": scaler,
        "metricas": metricas,
    }

    print(metricas)


Training model for 3comp
{'mae': 0.8761897751598245, 'medae': 0.7448976919284106, 'r2': 0.2918977034987047}

Training model for 4comp
{'mae': 0.9962637603410521, 'medae': 0.8204098233608748, 'r2': 0.42398071614004407}


In [43]:
for grupo in ["3comp", "4comp"]:

    rf = modelos[grupo]["rf"]
    scaler = modelos[grupo]["scaler"]

    sub = (
        df_feat[
            df_feat["grupo"] == grupo
        ]
        .copy()
    )

    X = scaler.transform(
        sub[FEATURE_COLS]
    )

    sub["pred_minutos"] = np.expm1(
        rf.predict(X)
    )

    predicciones.append(sub)

df_pred = pd.concat(
    predicciones,
    ignore_index=True,
)

In [44]:
for grupo in ["3comp", "4comp"]:

    sub = df_pred[
        df_pred["grupo"] == grupo
    ]

    mae = mean_absolute_error(
        sub["minutos_hasta_setpoint"],
        sub["pred_minutos"],
    )

    print(f"\n{grupo}")
    print(
        f"Events: {len(sub):,}"
    )
    print(
        f"MAE: {mae:.2f} min"
    )
    print(
        f"Median real: "
        f"{sub['minutos_hasta_setpoint'].median():.1f}"
    )
    print(
        f"Median pred: "
        f"{sub['pred_minutos'].median():.1f}"
    )


3comp
Events: 9,731
MAE: 15.67 min
Median real: 8.0
Median pred: 9.4

4comp
Events: 19,981
MAE: 28.69 min
Median real: 14.0
Median pred: 12.8


In [50]:
MIN_EVENTOS = 500
MIN_SOURCES_SPLIT = 2

conteos = (
    df_feat
    .groupby("source")
    .size()
    .sort_values(ascending=False)
)

fuentes_modelables = conteos[
    conteos >= MIN_EVENTOS
].index

print(f"Sources with >= {MIN_EVENTOS} events:")
print(conteos.loc[fuentes_modelables])

modelos_por_source = {}

for source in fuentes_modelables:
    df_source = df_feat[
        df_feat["source"] == source
    ].copy()

    print(
        f"\nTraining {source}"
        f" ({len(df_source)} events)"
    )

    rf, scaler, metricas = entrenar_rf(
        df_source
    )

    modelos_por_source[source] = {
        "modelo": rf,
        "scaler": scaler,
        "metricas": metricas,
        "n_eventos": len(df_source),
    }

    print(metricas)

Sources with >= 500 events:
source
POX21    3824
POX59    2066
POX55    1530
POX10    1192
POX41     998
POX40     911
POX46     902
POX9      890
POX43     886
POX58     878
POX28     849
POX56     830
POX60     803
POX54     740
POX24     698
POX57     669
POX50     662
POX37     650
POX48     616
POX29     606
POX20     590
POX61     504
dtype: int64

Training POX21 (3824 events)
{'mae': 0.7923789282554601, 'medae': 0.5833424577583521, 'r2': 0.5036276778525717}

Training POX59 (2066 events)
{'mae': 0.5870666146698159, 'medae': 0.434894457841552, 'r2': 0.5056495942834058}

Training POX55 (1530 events)
{'mae': 0.7690891881957485, 'medae': 0.6704060637834901, 'r2': 0.10893441279749061}

Training POX10 (1192 events)
{'mae': 0.8806234015635459, 'medae': 0.8321197426530991, 'r2': 0.2840289639340354}

Training POX41 (998 events)
{'mae': 0.8930298567265942, 'medae': 0.789682259049491, 'r2': 0.24349763645168143}

Training POX40 (911 events)
{'mae': 0.8425591840765501, 'medae': 0.744393311621

In [51]:
for source in sorted(df_pred["source"].unique()):

    sub = df_pred[
        df_pred["source"] == source
    ]

    mae = mean_absolute_error(
        sub["minutos_hasta_setpoint"],
        sub["pred_minutos"],
    )

    print(f"\n{source}")
    print(
        f"Events: {len(sub):,}"
    )
    print(
        f"MAE: {mae:.2f} min"
    )
    print(
        f"Median real: "
        f"{sub['minutos_hasta_setpoint'].median():.1f}"
    )
    print(
        f"Median pred: "
        f"{sub['pred_minutos'].median():.1f}"
    )


POX10
Events: 1,192
MAE: 22.73 min
Median real: 15.5
Median pred: 15.1

POX11
Events: 92
MAE: 4.05 min
Median real: 2.0
Median pred: 6.3

POX13
Events: 432
MAE: 43.77 min
Median real: 41.0
Median pred: 27.3

POX16
Events: 381
MAE: 26.89 min
Median real: 17.0
Median pred: 12.9

POX17
Events: 35
MAE: 6.10 min
Median real: 0.0
Median pred: 0.5

POX18
Events: 217
MAE: 36.60 min
Median real: 26.0
Median pred: 21.3

POX19
Events: 296
MAE: 5.41 min
Median real: 4.0
Median pred: 7.7

POX20
Events: 590
MAE: 50.79 min
Median real: 60.0
Median pred: 25.6

POX21
Events: 3,824
MAE: 35.59 min
Median real: 34.0
Median pred: 30.4

POX22
Events: 380
MAE: 16.77 min
Median real: 6.0
Median pred: 9.3

POX23
Events: 100
MAE: 25.43 min
Median real: 15.0
Median pred: 11.1

POX24
Events: 698
MAE: 13.84 min
Median real: 8.0
Median pred: 9.7

POX25
Events: 319
MAE: 21.95 min
Median real: 12.0
Median pred: 9.4

POX26
Events: 442
MAE: 22.30 min
Median real: 7.0
Median pred: 11.0

POX27
Events: 352
MAE: 38.09 min